# Local Neutralisation Profiles — Radial and Axial

Core-averaged density profiles and eta(z) from 3-D WarpX field arrays.

> Global ParticleNumber ratios are necessary but **not sufficient** to claim
> local space-charge compensation. This notebook provides spatial evidence.


In [ ]:
from pathlib import Path
import os, sys, subprocess, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Locate project root
_ROOT = Path.cwd()
while _ROOT.name != 'plasma_column' and _ROOT.parent != _ROOT:
    _ROOT = _ROOT.parent
if str(_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(_ROOT / 'src'))

WORK           = Path.home() / 'Work' / 'simulation_codes-working'
WARPX_DATA_DIR = WORK / 'warpx-data'
RESULTS_DIR    = _ROOT / 'results'
RUNS_DIR       = _ROOT / 'results'
PLOTS_DIR      = _ROOT / 'plots'
RESULTS_DIR.mkdir(exist_ok=True)
PLOTS_DIR.mkdir(exist_ok=True)

os.environ['WARPX_DATA_DIR']  = str(WARPX_DATA_DIR)
os.environ['LD_LIBRARY_PATH'] = (
    str(WORK / 'warpx' / 'install' / 'lib') + ':'
    + os.environ.get('LD_LIBRARY_PATH', '')
)
print('Python :', sys.executable)
print('ROOT   :', _ROOT)
print('WarpX data:', WARPX_DATA_DIR)


In [ ]:
from plasma_column.notebook_utils import print_simulation_config
_DEFAULTS = {
    'plasma cell z-range [m]':  '0.00 - 0.20',
    'beam-core radius [mm]':    2.0,
    'r_max for profiles [mm]':  15.0,
    'radial bins':              60,
    'grid (synthetic)':         '31 x 31 x 50',
    'eta_target (synthetic)':   0.70,
}
print_simulation_config(
    notebook_title='Local Neutralisation Profiles',
    defaults=_DEFAULTS, overrides={},
)


## 1. 3-D density arrays (synthetic demo)

Replace `ne_3d, ni_3d, np_3d, x, y, z` with arrays from a WarpX plotfile
(e.g. via `yt`) for production use.


In [ ]:
from plasma_column.plotting import (
    setup_publication_style, plot_radial_density_profile, plot_neutralization_vs_z,
)
from plasma_column.diagnostics import (
    generate_synthetic_3d_grid,
    compute_radial_density_profiles,
    compute_local_neutralization_vs_z,
    compute_local_core_neutralization,
)
setup_publication_style()

ETA_TARGET = 0.70
R_CORE_M   = 0.002

ne_3d, ni_3d, np_3d, x, y, z = generate_synthetic_3d_grid(
    nx=31, ny=31, nz=50, n_proton_peak=1e15, eta_target=ETA_TARGET,
)
print('Grid shape:', ne_3d.shape, '  z:', f'{z[0]:.3f} - {z[-1]:.3f} m')


## 2. Radial density profiles


In [ ]:
radial_df = compute_radial_density_profiles(
    ne_3d, ni_3d, np_3d, x, y, z,
    z_min_col=0.0, z_max_col=0.20, r_max=0.015, n_bins=60,
)
p, _ = plot_radial_density_profile(
    radial_df, PLOTS_DIR,
    case_name='local_profiles_demo', highlight_core_r=R_CORE_M,
)
plt.show()
print('Saved:', p.name)
display(radial_df.head())


## 3. Axial neutralisation profile eta(z)


In [ ]:
z_df = compute_local_neutralization_vs_z(
    ne_3d, ni_3d, np_3d, x, y, z, r_core=R_CORE_M,
)
p, _ = plot_neutralization_vs_z(
    z_df, PLOTS_DIR, case_name='local_profiles_demo', z_col_range=(0.0, 0.20),
)
plt.show()
print('Saved:', p.name)
display(z_df[['z','eta_electron_only_local_z','eta_net_local_z','keff_over_k0_local_z']].head(10))


## 4. Core-volume summary


In [ ]:
core = compute_local_core_neutralization(
    ne_3d, ni_3d, np_3d, x, y, z,
    z_min_col=0.0, z_max_col=0.20, r_core=R_CORE_M,
)
print('Core-averaged diagnostics:')
for k, v in core.items():
    print(f'  {k:<38} {v:.4g}')


## 5. 2-D transverse density slice (x–y cross-section at beam midplane)

False-color map of electron number density n_e(x, y) at the axial midplane z = L/2.
Beam-core region highlighted with a dashed circle.

In [ ]:
from plasma_column.diagnostics import generate_synthetic_3d_grid, compute_local_core_neutralization
from plasma_column.plotting import setup_publication_style
setup_publication_style()

# Re-use synthetic arrays from §1 if already defined, else regenerate
try:
    _ = ne_3d
except NameError:
    ne_3d, ni_3d, np_3d, x, y, z = generate_synthetic_3d_grid(
        nx=32, ny=32, nz=128,
        x_range=(-0.015, 0.015), y_range=(-0.015, 0.015), z_range=(0.0, 0.20),
        r_beam=0.002, n_beam_peak=1e13, n_plasma_peak=0.7e13,
    )

# Midplane slice
iz_mid = ne_3d.shape[2] // 2
ne_slice = ne_3d[:, :, iz_mid]   # shape (ny, nx)
np_slice = np_3d[:, :, iz_mid]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
titles  = [r"$ (electrons)", r"$ (beam protons)", r"Local $\eta = n_e / n_p$"]
cmaps   = ["plasma", "Blues", "RdYlGn"]
arrays  = [ne_slice, np_slice,
           np.where(np_slice > 0, np.clip(ne_slice / np_slice, 0, 1.2), 0)]
units   = ["m$^{-3}$", "m$^{-3}$", ""]

x_mm = x * 1e3
y_mm = y * 1e3
for ax, arr, title, cmap, unit in zip(axes, arrays, titles, cmaps, units):
    pcm = ax.pcolormesh(x_mm, y_mm, arr, cmap=cmap, shading="auto")
    cb = fig.colorbar(pcm, ax=ax, fraction=0.046, pad=0.04)
    cb.set_label(unit, fontsize=9)
    # Beam-core circle
    theta = np.linspace(0, 2*np.pi, 200)
    ax.plot(2.0 * np.cos(theta), 2.0 * np.sin(theta),
            "w--", lw=1.5, label="Beam core (r=2 mm)")
    ax.set_xlabel("x [mm]", fontsize=11)
    ax.set_ylabel("y [mm]", fontsize=11)
    ax.set_title(title, fontsize=12)
    ax.set_aspect("equal")

axes[0].legend(fontsize=8, loc="upper right")
plt.suptitle(r"Transverse Density Slice at Midplane ( = L/2$) — Synthetic Demo",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "transverse_density_slice.png", dpi=150, bbox_inches="tight")
plt.savefig(PLOTS_DIR / "transverse_density_slice.pdf", bbox_inches="tight")
plt.show()
print("Saved transverse_density_slice.{png,pdf}")

## 6. Axial neutralization profile — H2 vs Kr comparison (synthetic)

Side-by-side η(z) profiles: H2 saturates quickly along the column, Kr more slowly due to lower σ.
Uses analytic density profile (Gaussian beam, uniform gas) as a synthetic model.

In [ ]:
from plasma_column.neutralization import (
    gas_density_m3, ionization_tau_s, neutralization_fraction, proton_beta_gamma_speed
)
try:
    sigma_h2
except NameError:
    from plasma_column.gas import get_h2_cross_section, get_kr_cross_section
    sigma_h2 = get_h2_cross_section(30.0)
    sigma_kr = get_kr_cross_section(30.0)
    beta, gamma, v_beam = proton_beta_gamma_speed(30.0)

# Analytic: eta(z) = 1 - exp(-z / (v * tau)) for particle travelling along z
z_col = np.linspace(0, 0.20, 300)  # [m]

gas_configs = [
    ("H2", 1e-5, sigma_h2, "tab:blue"),
    ("H2", 3e-5, sigma_h2, "cornflowerblue"),
    ("Kr", 1e-6, sigma_kr, "tab:orange"),
    ("Kr", 3e-6, sigma_kr, "sandybrown"),
]

fig, ax = plt.subplots(figsize=(9, 5))
for gas, p_torr, sigma, col in gas_configs:
    n_gas = gas_density_m3(p_torr)
    tau   = ionization_tau_s(n_gas, sigma, v_beam)
    t_z   = z_col / v_beam   # transit time to reach z [s]
    eta_z = neutralization_fraction(t_z, tau, eta_ss=1.0)
    ax.plot(z_col * 100, eta_z, color=col, lw=2,
            label=f"{gas} {p_torr:.0e} Torr  (τ={tau*1e9:.0f} ns)")

ax.axhline(0.5, ls=":",  color="gray", lw=1.2, label="η = 0.5")
ax.axhline(0.9, ls="--", color="gray", lw=1.2, label="η = 0.9")
ax.axvline(20.0, ls="-.", color="black", lw=1.3, alpha=0.5, label="Column exit (20 cm)")

ax.set_xlabel("Axial Position $ [cm]", fontsize=12)
ax.set_ylabel(r"Local Neutralization $\eta(z)$", fontsize=12)
ax.set_title("Axial Neutralization Build-up Along Column — H₂ vs Kr", fontsize=13)
ax.legend(fontsize=8, ncol=2)
ax.grid(True, ls="--", alpha=0.4)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "eta_z_h2_kr_comparison.png", dpi=150, bbox_inches="tight")
plt.savefig(PLOTS_DIR / "eta_z_h2_kr_comparison.pdf", bbox_inches="tight")
plt.show()
print("Saved eta_z_h2_kr_comparison.{png,pdf}")

## 7. Phase-space portrait (synthetic transverse beam distribution)

Scatter plot of (x, p_x) with 1-σ RMS ellipse before and after partial space-charge compensation.
Compensation reduces the effective focusing defect, leading to a smaller halo.

In [ ]:
from plasma_column.plotting.transport import plot_phase_space

rng = np.random.default_rng(42)

# Uncompensated beam (large divergence due to space charge)
N = 8000
sigma_x_uncomp  = 3.5e-3   # [m] — blown up by space charge
sigma_px_uncomp = 12.0e-3  # [rad] — large divergence
rho_uncomp      = 0.35

# Compensated beam (eta=0.7, tighter phase space)
sigma_x_comp  = 2.0e-3  # [m]
sigma_px_comp = 6.5e-3  # [rad]
rho_comp      = 0.15

def sample_beam(sigma_x, sigma_px, rho, N, rng):
    cov = np.array([[sigma_x**2, rho*sigma_x*sigma_px],
                    [rho*sigma_x*sigma_px, sigma_px**2]])
    return rng.multivariate_normal([0, 0], cov, N)

pts_uncomp = sample_beam(sigma_x_uncomp, sigma_px_uncomp, rho_uncomp, N, rng)
pts_comp   = sample_beam(sigma_x_comp, sigma_px_comp, rho_comp, N, rng)

# Convert to mm / mrad for plotting
x_u,  px_u  = pts_uncomp[:, 0]*1e3, pts_uncomp[:, 1]*1e3
x_c,  px_c  = pts_comp[:, 0]*1e3,   pts_comp[:, 1]*1e3

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), sharey=False)
for ax, xp, pp, title, col in zip(
    axes,
    [x_u, x_c], [px_u, px_c],
    ["Uncompensated ($\eta=0$)", r"Compensated ($\eta=0.7$)"],
    ["tab:red", "tab:blue"]):
    ax.scatter(xp, pp, s=1.5, alpha=0.25, color=col, rasterized=True,
               label=f"N={N:,} particles")
    from matplotlib.patches import Ellipse
    cov = np.cov(xp, pp)
    sx, sp = np.sqrt(cov[0,0]), np.sqrt(cov[1,1])
    rho_plot = cov[0,1] / (sx * sp + 1e-30)
    ang = 0.5 * np.degrees(np.arctan2(2*rho_plot*sx*sp, sx**2 - sp**2))
    eps = np.sqrt(max(np.linalg.det(cov), 0))
    ell = Ellipse(xy=(np.mean(xp), np.mean(pp)), width=2*sx, height=2*sp,
                  angle=ang, edgecolor="black", facecolor="none", lw=2, ls="--")
    ax.add_patch(ell)
    ax.text(0.03, 0.97, f"$\varepsilon_{{\rm rms}}={eps:.2f}$ mm·mrad",
            transform=ax.transAxes, va="top", fontsize=9)
    ax.set_xlabel("$ [mm]", fontsize=12)
    ax.set_ylabel("/p_0$ [mrad]", fontsize=12)
    ax.set_title(title, fontsize=13)
    ax.legend(fontsize=9, loc="lower right", markerscale=4)
    ax.grid(True, ls="--", alpha=0.4)

plt.suptitle("Transverse Phase Space Portrait — 30 keV Proton Beam (Synthetic)",
             fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "phase_space_before_after.png", dpi=150, bbox_inches="tight")
plt.savefig(PLOTS_DIR / "phase_space_before_after.pdf", bbox_inches="tight")
plt.show()
print("Saved phase_space_before_after.{png,pdf}")